# 35. Scikit-Learn Capstone Mini-Project

## 🏢 End-to-End Enterprise Customer Churn & Lifetime Risk Predictive Pipeline

An industry-grade machine learning project demonstrating the complete data science lifecycle:
1. **Data Ingestion & Problem Definition**: Predicting customer subscription churn on a heterogeneous tabular portfolio
2. **Data Hygiene & Preprocessing Architecture**: Assembling a `ColumnTransformer` with `SimpleImputer`, `StandardScaler`, and `OneHotEncoder`
3. **Preventing Data Leakage**: Enforcing strict `train_test_split` partitioning
4. **Algorithm Benchmarking**: Training and comparing 4 candidate estimators:
   - Baseline: Logistic Regression
   - Non-Parametric: Random Forest Classifier
   - Margin-Based: Support Vector Classifier (RBF)
   - Sequential Ensemble: Gradient Boosting Classifier
5. **Hyperparameter Optimization**: Automated tuning using `GridSearchCV`
6. **Comprehensive Model Evaluation**: Confusion Matrix, ROC-AUC Curves, Precision-Recall trade-off, and Feature Importance profiling
7. **Model Deployment Readiness**: Serializing the complete pipeline to disk via `joblib`

### 1. Synthesizing Enterprise Customer Dataset Matrix

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)
n_samples = 1200

tenure_months = np.random.exponential(scale=24, size=n_samples).clip(1, 72)
monthly_charges = np.random.uniform(20, 120, size=n_samples)
total_charges = tenure_months * monthly_charges + np.random.normal(0, 50, size=n_samples)
contract_types = np.random.choice(['Month-to-month', 'One year', 'Two year'], size=n_samples, p=[0.55, 0.25, 0.20])
payment_methods = np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], size=n_samples)
tech_support = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.35, 0.65])

# Churn probability higher with high charges, month-to-month contract, and no tech support
churn_logits = -1.2 + 0.02 * (monthly_charges - 60) - 0.04 * (tenure_months - 20) + \
               (contract_types == 'Month-to-month') * 0.8 - (tech_support == 'Yes') * 0.6
churn_probs = 1 / (1 + np.exp(-churn_logits))
churn = (np.random.rand(n_samples) < churn_probs).astype(int)

df_telecom = pd.DataFrame({
    'Tenure': tenure_months.round(1),
    'Monthly_Charges': monthly_charges.round(2),
    'Total_Charges': total_charges.round(2),
    'Contract': contract_types,
    'Payment_Method': payment_methods,
    'Tech_Support': tech_support,
    'Churn': churn
})

# Introduce deliberate missing values for preprocessing realism
df_telecom.loc[np.random.choice(df_telecom.index, 20), 'Total_Charges'] = np.nan

print("Dataset Matrix Loaded. Shape:", df_telecom.shape)
print("Overall Churn Rate:", f"{df_telecom['Churn'].mean() * 100:.2f}%")
df_telecom.head(5)

### 2. Feature Partitioning & Preprocessing Pipeline Assembly

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df_telecom.drop(columns='Churn')
y = df_telecom['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

numeric_cols = ['Tenure', 'Monthly_Charges', 'Total_Charges']
categorical_cols = ['Contract', 'Payment_Method', 'Tech_Support']

# Numeric Transformer
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Transformer
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Composite ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('cat', cat_pipe, categorical_cols)
])

print("Preprocessing Pipeline assembled successfully!")

### 3. Multi-Model Benchmark & Cross-Validation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector (RBF)': SVC(probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

benchmark_results = {}
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    benchmark_results[name] = (scores.mean(), scores.std())
    print(f"{name:22s} -> Mean ROC-AUC: {scores.mean():.4f} (± {scores.std():.4f})")

### 4. Hyperparameter Tuning on Top Candidate (Gradient Boosting)

In [ ]:
from sklearn.model_selection import GridSearchCV

# Full pipeline with top candidate
tuning_pipe = Pipeline([
    ('prep', preprocessor),
    ('gb', GradientBoostingClassifier(random_state=42))
])

param_grid = {
    'gb__n_estimators': [60, 100],
    'gb__learning_rate': [0.05, 0.10],
    'gb__max_depth': [2, 3]
}

grid_search = GridSearchCV(tuning_pipe, param_grid, cv=4, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Optimal Parameters: {grid_search.best_params_}")
print(f"Best CV ROC-AUC:    {grid_search.best_score_:.4f}")

### 5. Final Holdout Evaluation & Diagnostics

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_probs = best_model.predict_proba(X_test)[:, 1]

print("=== Final Holdout Test Performance ===")
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))
print(f"Holdout ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")

# Visual Diagnostic Panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=100)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Retained', 'Churned'], cmap='Blues', colorbar=False, ax=ax1)
ax1.set_title("1. Confusion Matrix", fontweight='bold')

RocCurveDisplay.from_estimator(best_model, X_test, y_test, color='#2980b9', lw=2, ax=ax2)
ax2.plot([0, 1], [0, 1], 'k--', lw=1)
ax2.set_title("2. Holdout ROC Curve", fontweight='bold')

plt.tight_layout()
plt.show()

### 6. Production Model Persistence via Joblib

In [ ]:
import joblib
import os

# Serialize full production pipeline
pipeline_path = 'customer_churn_production_pipeline.joblib'
joblib.dump(best_model, pipeline_path)
print(f"Production Pipeline successfully serialized: '{pipeline_path}' ({os.path.getsize(pipeline_path)} bytes)")

# Verify Reloaded Inference on Raw Record
loaded_pipeline = joblib.load(pipeline_path)
raw_test_customer = pd.DataFrame({
    'Tenure': [3.5],
    'Monthly_Charges': [89.50],
    'Total_Charges': [313.25],
    'Contract': ['Month-to-month'],
    'Payment_Method': ['Electronic check'],
    'Tech_Support': ['No']
})

churn_prob = loaded_pipeline.predict_proba(raw_test_customer)[0, 1]
print(f"Inference Verification: Customer Churn Probability = {churn_prob * 100:.1f}%")
print(f"Risk Tier: {'HIGH RISK' if churn_prob > 0.5 else 'LOW RISK'}")

# Cleanup demonstration artifact
if os.path.exists(pipeline_path): os.remove(pipeline_path)